# Research Diary 2

## 1

In the week of my research diary 1, I solidified my understanding of what a minimally sufficient representation is supposed to capture. I also touched on nuisance variables (e.g., rotation, scale, lighting, viewpoint) and how they can prevent a representation from being minimal.

Given that, my goal for this week was to read the paper fully and understand how to minimize the impact of nuisances using Sampled Anti-Aliased Likelihood (SAL) and SIFT/DSP-SIFT, and how these ideas connect to CNN pooling.

While I didn’t finish the entire paper, I did develop a solid understanding of several methods that make $y$ more nuisance-invariant.

## 2

Papers:
* Visual Representations: Defining Properties and Deep Approximations, S. Soatto, A. Chiuso, ICLR 2016; [Web Link](https://arxiv.org/pdf/1411.7676v9), [Github Link](../Papers/Visual_Representations_defining_properties_and_deep_approximations.pdf)
  * Time Spent: 7.5 hrs

Videos:
* [Pavel Novikov - Building Intuition for Sample Rate and Aliasing](https://www.youtube.com/watch?v=SjimjNDEN8c)
  * Time Spent: 0.5 hrs

Blogs:
* [Introduction to SIFT( Scale Invariant Feature Transform)](https://medium.com/@deepanshut041/introduction-to-sift-scale-invariant-feature-transform-65d7f3a72d40)
  * Time Spent: 0.5 hrs
* [Link](https://chatgpt.com/share/68e96c47-4338-8010-a3ed-e58ef58a0ebb) to AI Transcript


## 3

* SAL (Sampled Anti-Aliased Likelihood).
  * The nuisance space G (rotation/scale/translation/contrast, etc.) is continuous; we sample it at ${g_i} to make the problem tractable.
  * For each sample $g_i$, we anti-alias (i.e., locally average) the likelihood over tiny offsets near $g_i$ to avoid grid artifacts: local mean-pooling in transform space.
  * We then pool across samples (often with a max) to approximate the ideal profile likelihood $sup_{g \in G}p_{\theta, g}(y)$.
  * The $\varepsilon$ in the claim is a target approximation error: smaller $\varepsilon$ ⇒ denser sampling and tighter averaging windows.
* Profile vs. marginal likelihoods.
  * Profile: treat nuisance as unknown but deterministic → maximize over $g$.
  * Marginal: treat nuisance as random → average $\int p_{\theta}(y|g)dP(g)$
  * “Average out” $\neq$ “eliminate”; it means integrate with a (possibly local) measure.
  * SAL is a practical bridge: sample + locally average + max.
* Sampling $G$ & anti-aliasing.
  * “Sampling $G$” = choosing a finite menu of representative transforms (e.g., rotations every $15$&deg;, a few scales).
  * Local averaging around each sample reduces aliasing from coarse grids; the weights are stability kernels (often normalized so they act like local priors).
* Invariance, insensitivity, selectivity
  * Insensitive (stable): small changes in $g$ → small changes in output (still depends on $g$).
  * Invariant: output is constant w.r.t. $g$.
  * Maximal invariant (“selective”): equal outputs imply inputs differ only by $g$.
  * The paper’s target is a sufficient invariant for the task, not necessarily maximal/invertible.
* Canonization vs Pooling
  * Canonization: estimate nuisance (e.g., dominant orientation/scale) and undo it to a canonical frame before describing the patch.
  * Pooling (marginal/profile/SAL): integrate or maximize over plausible nuisances without committing to a single estimate.
  * SIFT combines both: canonize (orientation/scale) and pool (spatial Gaussian window, soft orientation bins).
* SIFT / DSP-SIFT internals.
  * Gradient orientation = direction of local intensity change, $\theta = atan2(L_y,L_x)$; magnitude $= \sqrt{L_x^2 + L_y^2}$
  * Build orientation histograms within a 4×4 spatial grid (cells), each with 8 orientation bins → 128-D descriptor; soft binning (Parzen kernels) and a spatial Gaussian window provide anti-aliasing.
  * DSP-SIFT adds domain-size pooling (average over nearby patch sizes) for scale jitter robustness—another SAL-style anti-alias in scale.
* In-plane vs. out-of-plane rotations & occlusion.
  * In-plane rotations are handled well by SIFT (via orientation canonization).
  * Out-of-plane (3D tilts) introduce projective/affine changes; small tilts are tolerable, large ones are harder (ASIFT or sampling tilts can help).
  * Occlusion motivates local representations (patches/receptive fields) so invisible parts don’t corrupt the whole representation.
* How this connects to CNNs (feature maps ↔ filters).
  * Break the image into receptive fields $V_j = g_jB_0$; score local likelihoods $p(y|v_j | \theta_k, g_ig_j)$ → these are feature maps.
  * Combine them with weights $w_{jk}$ (implemented as filters/kernels in a conv layer) and pool across positions/classes.
  * Pushing transforms to the input and setting nuisance to the identity (using the group’s identity $e$) matches the equivariance view used by convolutions.

## 4

* I want to understand the CNN architecture connection more concretely (how Eq.-by-Eq. turns into conv, pooling, and multi-layer compositions). I plan to read the later sections next week and sketch a minimal prototype.
* I’m interested in reconstruction from minimally sufficient invariants. SAL/SIFT intentionally pool/average for invariance, which is information-losing, so reconstruction is unreliable. I haven’t investigated yet, but after first having understood how the concepts I have learnt connect with CNN, if I don't find a more interesting problem, I would like to investigate this angle

## 5 (N/A)

Given that the paper is primarily theoretical, I have not had a scenario to run experiments

## 6
Resolved
* Sampling rate & anti-aliasing in nuisance space (biggest hurdle). I initially struggled to map “sampling a group $G$” to something concrete. I now understand it as choosing a finite grid of nuisance settings (e.g., rotations, scales), then doing local averaging in transform space around each grid point to prevent aliasing from coarse sampling. The sampling rate (grid spacing) and the anti-alias window (kernel bandwidth) trade off approximation error vs. compute; tighter error targets $\varepsilon$ require denser sampling and smaller averaging windows.
* SIFT’s many normalization steps. The sequence—keypoint detection → canonization (center/scale/orientation) → spatial Gaussian window → orientation soft-binning → descriptor normalization—felt like a maze. I now see each step as addressing a nuisance axis: canonization for translation/scale/rotation; Gaussian window for local translation stability; soft-binning (Parzen) for orientation jitter; L2 norm/clipping for photometric robustness. DSP-SIFT adds domain-size pooling to anti-alias in scale as well.

Partly Resolved:
* CNN connection (weights as filters). The algebraic mapping (sum of feature maps weighted by $w_{jk}$) to conv + pooling is clear, but I want to implement a tiny example where the likelihood-style sum numerically equals a conv layer’s output.

## 7

* For next week I want to go deeper into the Deep Convolutional architecture section (Section 4) of the paper. I want to see how the minimally sufficient representation comes in play with CNN architecture and if there are any optimization opportunities there.
* I would also like to see what minimally sufficient representation would look like if reconstruction is to be possible
* By next week, I want my future research direction to be determined and some preliminary readings/experiments to have been run